In [1]:
pip install rapidfuzz

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 8.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [13]:
import json
import re
import pandas as pd
from pathlib import Path
from rapidfuzz import fuzz

# --------------------------------------------------
# 1. FILE PATHS
# --------------------------------------------------

BASE_DIR = Path("..").resolve()

GRAILED_FILE = BASE_DIR / "data" / "grailed" / "jordan.csv"
CATALOG_FILE = BASE_DIR / "data" / "lista_jordan.json"

OUTPUT_FILE = BASE_DIR / "data" / "processed" / "jordan_matched.csv"
OUTPUT_FILE_1 = BASE_DIR / "data" / "processed" / "jordan_matched-1.csv"

In [3]:
# --------------------------------------------------
# 2. LOAD DATA
# --------------------------------------------------

grailed_df = pd.read_csv(GRAILED_FILE)

with open(CATALOG_FILE, "r", encoding="utf-8") as f:
    catalog = json.load(f)

catalog_df = pd.DataFrame(catalog)

print("Grailed listings:", len(grailed_df))
print("JSON products:", len(catalog_df))

Grailed listings: 71824
JSON products: 1848


In [4]:
# --------------------------------------------------
# 3. NORMALIZE TEXT
# --------------------------------------------------

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Treat Air Jordan and Jordan as equivalent
    text = text.replace("air jordan", "jordan")

    # Remove punctuation and special characters
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_brand(brand):
    brand = normalize_text(brand)

    if "jordan" in brand:
        return "jordan"

    return brand


def extract_year(text):
    if pd.isna(text):
        return None

    match = re.search(r"\b(19\d{2}|20\d{2})\b", str(text))

    return int(match.group()) if match else None


# Prepare Grailed data
grailed_df["normalized_title"] = grailed_df["title"].apply(normalize_text)
grailed_df["normalized_brand"] = grailed_df["brand"].apply(normalize_brand)
grailed_df["grailed_year"] = grailed_df["title"].apply(extract_year)

# Prepare catalogue data
catalog_df["normalized_name"] = catalog_df["shoeName"].apply(normalize_text)
catalog_df["normalized_brand"] = catalog_df["brand"].apply(normalize_brand)
catalog_df["catalog_year"] = catalog_df["shoeName"].apply(extract_year)

In [5]:
# --------------------------------------------------
# 4. MATCHING FUNCTION
# --------------------------------------------------

def calculate_match_score(grailed_row, product):

    title = grailed_row["normalized_title"]
    product_name = product["normalized_name"]

    # Compare product titles
    token_score = fuzz.token_set_ratio(title, product_name)
    ratio_score = fuzz.WRatio(title, product_name)

    title_score = (token_score * 0.6) + (ratio_score * 0.4)

    # Additional signals
    score = title_score

    grailed_year = grailed_row["grailed_year"]
    catalog_year = product["catalog_year"]

    if grailed_year is not None and catalog_year is not None:
        if grailed_year == catalog_year:
            score += 5
        else:
            score -= 5

    # Compare colors when Grailed has a meaningful color
    grailed_color = normalize_text(grailed_row["color"])
    catalog_color = normalize_text(product["colorway"])

    generic_colors = {"", "multi", "multicolor", "unknown", "other"}

    if grailed_color not in generic_colors and catalog_color:
        color_score = fuzz.token_set_ratio(
            grailed_color,
            catalog_color
        )

        # Small bonus for compatible color descriptions
        if color_score >= 70:
            score += 3

    return min(score, 100)


# --------------------------------------------------
# 5. FIND TOP CANDIDATES
# --------------------------------------------------

def find_matches(grailed_row, top_n=3):

    # Restrict matching to the same brand
    candidates = catalog_df[
        catalog_df["normalized_brand"] == grailed_row["normalized_brand"]
    ]

    if candidates.empty:
        return []

    results = []

    for _, product in candidates.iterrows():

        score = calculate_match_score(grailed_row, product)

        results.append({
            "product": product,
            "score": round(score, 2)
        })

    # Highest score first
    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results[:top_n]

In [6]:
# --------------------------------------------------
# 6. MATCH ALL GRAILED LISTINGS
# --------------------------------------------------

match_results = []

for _, listing in grailed_df.iterrows():

    candidates = find_matches(listing, top_n=3)

    result = {
        "listing_id": listing["listing_id"],
        "grailed_title": listing["title"],
        "grailed_brand": listing["brand"],
        "grailed_color": listing["color"],
        "grailed_sold_price": listing["sold_price"],
    }

    # Best candidate
    if candidates:

        best = candidates[0]
        product = best["product"]

        second_score = (
            candidates[1]["score"]
            if len(candidates) > 1
            else 0
        )

        result.update({
            "matched_shoe": product["shoeName"],
            "matched_styleID": product["styleID"],
            "matched_product_id": product["_id"],
            "matched_colorway": product["colorway"],
            "release_date": product["releaseDate"],
            "retail_price": product["retailPrice"],
            "last_sale": product["lastSale"],
            "lowest_ask": product["lowestAsk"],
            "match_score": best["score"],
            "second_best_score": second_score,
            "match_gap": round(best["score"] - second_score, 2),
        })

        # Conservative matching thresholds
        if best["score"] >= 90 and best["score"] - second_score >= 5:
            result["match_status"] = "high_confidence"

        elif best["score"] >= 75:
            result["match_status"] = "needs_review"

        else:
            result["match_status"] = "no_reliable_match"

        # Save alternatives
        for i, candidate in enumerate(candidates, start=1):
            result[f"candidate_{i}"] = candidate["product"]["shoeName"]
            result[f"candidate_{i}_score"] = candidate["score"]

    else:
        result.update({
            "matched_shoe": None,
            "matched_styleID": None,
            "matched_product_id": None,
            "matched_colorway": None,
            "release_date": None,
            "retail_price": None,
            "last_sale": None,
            "lowest_ask": None,
            "match_score": 0,
            "second_best_score": 0,
            "match_gap": 0,
            "match_status": "no_candidates",
        })

    match_results.append(result)


# --------------------------------------------------
# 7. SAVE RESULTS
# --------------------------------------------------

matches_df = pd.DataFrame(match_results)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

matches_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("\nMatching completed!")
print("Output:", OUTPUT_FILE)

print("\nMatch status:")
print(matches_df["match_status"].value_counts())

display(
    matches_df[
        [
            "grailed_title",
            "matched_shoe",
            "match_score",
            "match_status",
            "candidate_2",
            "candidate_2_score"
        ]
    ].head(20)
)


Matching completed!
Output: C:\Proiecte\master\iapd\Resell_price_predictor_IAPD\data\processed\jordan_matched.csv

Match status:
match_status
needs_review         53241
high_confidence      11229
no_reliable_match     6393
no_candidates          961
Name: count, dtype: int64


,grailed_title,matched_shoe,match_score,match_status,candidate_2,candidate_2_score
0,Jordan 1 Old Love New Love Beginning Moments P...,Jordan 3 Retro Cool Grey (2007),72.20,no_reliable_match,Jordan 5 Retro Black Metallic (2007),71.01
1,"2001 Japan Grey Jordan 1 (11,535/50,000)",Jordan 1 Low Grey White,72.53,no_reliable_match,Jordan 2.0 Wolf Grey,71.78
2,Midnight Navy Blue Jordan 1,Jordan 1 Low Midnight Navy University Blue,92.20,needs_review,Jordan 1 Low Midnight Navy University Blue (GS),92.20
3,Jordan 10 Russell Westbrook Class of 2006,Jordan 5 Retro Olive (2006),75.87,needs_review,Jordan 5 Retro Grape (2006),75.87
4,Jordan 11 Low Concord Bred,Jordan 11 Retro Low Concord Bred (GS),93.00,high_confidence,Jordan 1 Low SE Concord,83.00
5,Vintage 2009 Jordan 6 Retro Reflective Sneaker...,Jordan 4 Retro Lakers,86.20,needs_review,Jordan 6 Retro Hare,83.11
6,Nike Jordan 11 Retro High Win Like ‘82 Blue White,Jordan 11 Retro UNC Win Like 82,88.06,needs_review,Jordan 11 Retro Win Like 96,85.67
7,Son of Mars,Jordan 3 Retro SE Dunk on Mars (GS),61.20,no_reliable_match,Jordan Mars 270 PSG Paris Saint-Germain,61.20
8,Size 13 - Jordan 14 “Hyper Royal”,Jordan 1 Retro High OG Hyper Royal,73.06,no_reliable_match,Jordan 1 Mid Hyper Royal Tumbled Leather,73.06
9,Air Jordan 5 Retro ‘Laney’ 2013,Jordan 5 Retro Laney (2013),100.00,needs_review,Jordan 5 Retro Grape (2013),96.00


In [14]:
matches_df = pd.DataFrame(match_results)

# Define the desired status order
status_order = [
    "high_confidence",
    "needs_review",
    "no_reliable_match",
    "no_candidates"
]

# Sort by matching status, then by match score (highest first)
matches_df["match_status"] = pd.Categorical(
    matches_df["match_status"],
    categories=status_order,
    ordered=True
)

matches_df = matches_df.sort_values(
    by=["match_status", "match_score"],
    ascending=[True, False],
    na_position="last"
)

# Create output folder if it doesn't exist
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# Export using semicolon delimiter for Excel
matches_df.to_csv(
    OUTPUT_FILE_1,
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

print(f"Saved {len(matches_df)} matches to: {OUTPUT_FILE_1}")


Saved 71824 matches to: C:\Proiecte\master\iapd\Resell_price_predictor_IAPD\data\processed\jordan_matched-1.csv


In [ ]:

import pandas as pd
from pathlib import Path

# Paths
BASE_DIR = Path("..").resolve()

GRAILED_FILE = BASE_DIR / "data" / "grailed" / "jordan.csv"
MATCH_FILE = BASE_DIR / "data" / "processed" / "jordan_matched.csv"
OUTPUT_FILE = BASE_DIR / "data" / "processed" / "jordan_training.csv"

# Load both datasets
grailed_df = pd.read_csv(GRAILED_FILE, sep=None, engine="python")
matches_df = pd.read_csv(
    MATCH_FILE,
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)
# Keep only high-confidence matches
high_confidence_df = matches_df[
    matches_df["match_status"] == "high_confidence"
].copy()

# Select useful catalogue columns
catalog_columns = [
    "listing_id",
    "matched_shoe",
    "matched_styleID",
    "matched_product_id",
    "matched_colorway",
    "release_date",
    "retail_price",
    "last_sale",
    "lowest_ask",
    "match_score"
]

high_confidence_df = high_confidence_df[catalog_columns]

# Avoid duplicate listings
high_confidence_df = high_confidence_df.drop_duplicates(
    subset="listing_id",
    keep="first"
)

# Combine original Grailed data with catalogue data
training_df = grailed_df.merge(
    high_confidence_df,
    on="listing_id",
    how="inner",
    validate="one_to_one"
)

# Convert both dates to timezone-naive datetime values
training_df["sold_at"] = pd.to_datetime(
    training_df["sold_at"], errors="coerce", utc=True
).dt.tz_localize(None)

training_df["release_date"] = pd.to_datetime(
    training_df["release_date"], errors="coerce", utc=True
).dt.tz_localize(None)

# Calculate product age when it was sold
training_df["days_since_release_at_sale"] = (
    training_df["sold_at"] - training_df["release_date"]
).dt.days

# Sort chronologically
training_df = training_df.sort_values(
    by="sold_at",
    ascending=True
)

# Export
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

training_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

print(f"High-confidence listings: {len(training_df)}")
print(f"Saved to: {OUTPUT_FILE}")

display(training_df.head(10))


TypeError: Cannot subtract tz-naive and tz-aware datetime-like objects

In [2]:
print("Matching CSV columns:")
print(matches_df.columns.tolist())

print("\nFirst 5 rows:")
display(matches_df.head())

print("Grailed CSV columns:")
print(grailed_df.columns.tolist())

Matching CSV columns:
['\ufefflisting_id', 'grailed_title', 'grailed_brand', 'grailed_color', 'grailed_sold_price', 'matched_shoe', 'matched_styleID', 'matched_product_id', 'matched_colorway', 'release_date', 'retail_price', 'last_sale', 'lowest_ask', 'match_score', 'second_best_score', 'match_gap', 'match_status', 'candidate_1', 'candidate_1_score', 'candidate_2', 'candidate_2_score', 'candidate_3', 'candidate_3_score']

First 5 rows:


,﻿listing_id,grailed_title,grailed_brand,grailed_color,grailed_sold_price,matched_shoe,matched_styleID,matched_product_id,matched_colorway,release_date,...,match_score,second_best_score,match_gap,match_status,candidate_1,candidate_1_score,candidate_2,candidate_2_score,candidate_3,candidate_3_score
0,104225454,Jordan 1 Old Love New Love Beginning Moments P...,Jordan Brand,multi,14,Jordan 3 Retro Cool Grey (2007),315297-062,6af0e1fd15a49eec462f68cc,Silver/Sport Red-Light Graphite-Orange Peel,2007-02-24,...,72.20,71.01,1.19,no_reliable_match,Jordan 3 Retro Cool Grey (2007),72.20,Jordan 5 Retro Black Metallic (2007),71.01,Jordan 5 Retro Fear Pack,66.91
1,104225464,"2001 Japan Grey Jordan 1 (11,535/50,000)",Jordan Brand,gray,10,Jordan 1 Low Grey White,553558-044,6af0dd1e15a49e7fc22f6485,Grey/White,2025-08-20,...,72.53,71.78,0.75,no_reliable_match,Jordan 1 Low Grey White,72.53,Jordan 2.0 Wolf Grey,71.78,Jordan 4 RM Wolf Grey,70.45
2,106218813,Midnight Navy Blue Jordan 1,Jordan Brand,blue,9,Jordan 1 Low Midnight Navy University Blue,553558-404,6af0dc7315a49e583c2f634e,Midnight Navy/White/University Blue,2026-03-20,...,92.20,92.20,0.00,needs_review,Jordan 1 Low Midnight Navy University Blue,92.20,Jordan 1 Low Midnight Navy University Blue (GS),92.20,Jordan ADG 5 White Midnight Navy University Blue,89.89
3,105954898,Jordan 10 Russell Westbrook Class of 2006,Jordan Brand,multi,15,Jordan 5 Retro Olive (2006),314259-381,6af0fb5f15a49e00bc2f6c8d,Army Olive/Solar Orange/Black,2006-11-18,...,75.87,75.87,0.00,needs_review,Jordan 5 Retro Olive (2006),75.87,Jordan 5 Retro Grape (2006),75.87,Jordan 12 Retro Chris Paul Class Of 2003,67.59
4,101144645,Jordan 11 Low Concord Bred,Jordan Brand,multi,16,Jordan 11 Retro Low Concord Bred (GS),528896-160,6af0dd0215a49e70ed2f6451,White/Gym Red-Black-True Red,2020-06-27,...,93.00,83.00,10.00,high_confidence,Jordan 11 Retro Low Concord Bred (GS),93.00,Jordan 1 Low SE Concord,83.00,Jordan 4 RM Bred,78.09


Grailed CSV columns:
['listing_id', 'brand', 'title', 'category', 'subcategory', 'size', 'condition', 'color', 'country', 'sold_price', 'last_asking_price', 'first_price', 'n_price_drops', 'followers', 'listed_at', 'sold_at', 'days_on_market']
